# MobileNetV2 op CIFAR-10 · vijf-epochpilot

**Doel:** vroege leerprogressie en praktische uitvoerbaarheid beoordelen voor W/A 32, 8, 4, 2 en 1.

Dit notebook leest uitsluitend de opgeslagen rapporten. Het start geen training of evaluatie en verandert geen checkpoints. Iedere complete epoch gebruikt 50.000 trainingsbeelden en 10.000 testbeelden. De latere 400-epochreferentie begint opnieuw vanaf nul.

**Model:** de besproken MobileNetV2-reconstructie met K=3: acht kanalen per centrale convolutiegroep, batch 128, Adam lr 0,001, FP32-masters en vijf BatchNorm-banken per positie. Dit is nog geen MX-proef en geen bevestigde exacte auteursimplementatie. Zie `PROTOCOL.md` voor de quantisatiepunten en FP-uitzonderingen.


## 1. Rapport kiezen

Laat `RUN_DIR = None` om het nieuwste rapport te gebruiken. Ook een nieuw **FAIL-, PAUSED- of onderbroken rapport** wordt gekozen; een oudere PASS wordt niet stilzwijgend voorgesteld als de nieuwste uitkomst.

Vul `RUN_DIR` in voor een specifieke resultatenmap of een uitgepakte feedbackbundel. Gebruik zo nodig `PROJECT_ROOT` als de projectmap niet vanuit de notebooklocatie gevonden wordt.


In [ ]:
from pathlib import Path
import importlib.util
import json
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

PROJECT_ROOT = None  # bijvoorbeeld Path(r"C:\Users\renzo\master-thesis-mx")
RUN_DIR = None       # bijvoorbeeld Path(r"C:\Users\renzo\master-thesis-mx\mx_runs\anyprecision_cifar10_mobilenetv2_k3_pilot5\..._run")

PACKAGE = "anyprecision_cifar10_mobilenetv2_k3_pilot5_v1"
EXPERIMENT = "anyprecision_cifar10_mobilenetv2_k3_pilot5"
roots = []
for origin in [Path.cwd()] + ([Path(PROJECT_ROOT)] if PROJECT_ROOT else []) + ([Path(RUN_DIR)] if RUN_DIR else []):
    origin = origin.parent if origin.is_file() else origin
    roots.extend([origin, *list(origin.parents)[:5]])
candidates = []
for root in dict.fromkeys(roots):
    candidates.extend([root / "notebook_support.py",
                       root / "runner_source" / "notebook_support.py",
                       root / "software_checks" / PACKAGE / "notebook_support.py",
                       root / PACKAGE / "notebook_support.py"])
helper = None
for candidate in candidates:
    config = candidate.parent / "experiment_config.json"
    if candidate.is_file() and config.is_file():
        if json.loads(config.read_text(encoding="utf-8")).get("experiment") == EXPERIMENT:
            helper = candidate.resolve()
            break
if helper is None:
    raise FileNotFoundError("notebook_support.py van deze pilot niet gevonden. Open de notebook in zijn pakketmap, of vul PROJECT_ROOT / RUN_DIR in.")
spec = importlib.util.spec_from_file_location("mobilenet_pilot5_notebook_support", helper)
ns = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ns)

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10,
                     "axes.labelcolor": "#263D4B", "axes.titlecolor": "#263D4B",
                     "axes.titlesize": 12, "figure.facecolor": "white", "savefig.dpi": 160})
context = ns.load_run(run_dir=RUN_DIR, project_root=PROJECT_ROOT)
display(ns.overview_table(context))
for note in ns.status_notes(context):
    display(Markdown("- " + note))


## 2. Accuracy over alle gerapporteerde epochs

Ruwe testaccuracies, zonder smoothing. Alle vijf modi worden na elke complete epoch op dezelfde volledige testset geëvalueerd. De verticale markering wijst de epoch met de hoogste **W32/A32**-accuracy aan; bij een exacte tie blijft de vroegste epoch gekozen.

Een verschil tijdens vijf vroege epochs kan mede een verschil in leersnelheid zijn. Het is nog geen schatting van het uiteindelijke precisieverlies.


In [ ]:
test_metrics = ns.epoch_metrics_table(context, "validation")
if test_metrics.empty:
    print("Geen complete epochs beschikbaar voor een accuracygrafiek.")
else:
    display(test_metrics.pivot(index="Epoch", columns="Modus", values="Top-1 [%]").reindex(columns=[f"W{b}/A{b}" for b in ns.BITS]).round(3))
    ns.plot_accuracy(context)
    plt.show()


## 3. Training en test per precisie

De trainingsaccuracy is het gemiddelde tijdens de gewichtsupdates en gebruikt geaugmenteerde beelden. De testaccuracy wordt daarna gemeten met vaste gewichten en zonder augmentatie. Het zijn dus verschillende meetmomenten en invoercondities.


In [ ]:
figure = ns.plot_train_validation(context)
if figure is not None:
    plt.show()


## 4. Laatste epoch en geselecteerd checkpoint

De laatste **werkelijk voltooide** epoch staat naast de epoch die op W32/A32 is geselecteerd. Bij een volledige pilot is de laatste epoch 5. De accuracies van de andere modi komen steeds uit **hetzelfde geselecteerde checkpoint**; zij kiezen niet ieder hun eigen beste epoch.

De selectie gebruikt hier de officiële testset, zoals in de afgesproken rapportage. De geselecteerde beste score is daarom geen onafhankelijke nieuwe testscore. De vooraf vastgelegde laatste-epochscore blijft zichtbaar. Wanneer beide keuzes dezelfde epoch zijn, zijn de twee kolommen terecht gelijk.


In [ ]:
selected = ns.selection_table(context)
if not selected.empty:
    display(selected[["Selectie", "Epoch", "Modus", "Test top-1 [%]", "Test top-5 [%]"]].round(3))
    ns.plot_selection(context)
    plt.show()


## 5. Loss en eerste leerprogressie

**Training:** W32 gebruikt harde cross-entropy; W8, W4, W2 en W1 gebruiken recursieve distillatiedoelen. Hun trainingslosses zijn daardoor niet onderling als dezelfde kwaliteitsmaat te rangschikken. **Test:** alle vijf modi gebruiken gewone cross-entropy met labels.

Volg vooral of elke modus vooruitgaat. Een lage vroege accuracy is op zichzelf geen bewijs dat die modus uiteindelijk niet kan leren.


In [ ]:
figure = ns.plot_losses(context)
if figure is not None:
    plt.show()


## 6. Gemeten tijd en geheugen per epoch

De epochtijd bevat training, evaluatie, checkpointopslag en de gemeten overige epochkosten. De eerste epoch kan extra opwarming bevatten. Bij hervatting kunnen historische tijden ontbreken wanneer alleen een checkpoint zonder zijn rapportbijlage beschikbaar is; zulke waarden blijven leeg.

Geheugenpieken worden opnieuw gemeten per epoch. **Gealloceerd geheugen is onderdeel van gereserveerd geheugen; tel ze niet op.** Dit zijn PyTorch-allocatortellers, geen directe meting van Windows' gedeelde GPU-geheugen of fysieke residentie.


In [ ]:
display(ns.timing_table(context).round(3))
figure = ns.plot_epoch_times(context)
if figure is not None:
    plt.show()
display(ns.memory_table(context).round(3))
figure = ns.plot_memory(context)
if figure is not None:
    plt.show()


## 7. Raming voor langere runs

De raming gebruikt uitsluitend complete bekende epochtijden. Als meer dan één complete tijdmeting beschikbaar is, wordt epoch 1 vanwege de eerste opwarming weggelaten. De tabel vermeldt precies welke epochs de basis vormen. Er wordt niet opnieuw met vijf vermenigvuldigd: alle vijf precisies zitten al in de gemeten tijd.

De projecties zijn **geen uitgevoerde trainingen**. Zij omvatten dezelfde volledige evaluatie en één checkpoint per epoch, maar geen eenmalige download of procesopstart. Vijf epochs bieden nog geen garantie voor eenzelfde tempo gedurende uren; thermiek en andere taken kunnen de duur beïnvloeden.


In [ ]:
forecast = ns.forecast_table(context)
if forecast.empty:
    print("Geen volledige epochtijd beschikbaar voor een totale raming.")
else:
    display(forecast.round(3))
    ns.plot_forecast(context)
    plt.show()


## 8. Wat nemen we mee naar de volgende stap?

- Zijn alle vijf modi numeriek stabiel en neemt hun testaccuracy toe?
- Blijven de 1- en 2-bitmodi langdurig rond kansniveau, of begint hun leren ook zichtbaar?
- Blijven epochduur en geheugenpieken ongeveer gelijk?
- Wat betekenen de gemeten tijden voor sessies van 50 of 100 epochs?

Deze pilot beslist nog niet of de accuracies geconvergeerd zijn of hoeveel epochs daarvoor nodig zijn. De volledige referentie begint later opnieuw vanaf nul. De uitkomsten en bronkeuzes blijven terug te vinden in de rapporten en `PROTOCOL.md`.
